# Forecast Validation Walkthrough

This notebook is the glass-box companion to the multi-horizon forecasting work.

Goals:
- confirm the validation method is chronological rolling-origin / walk-forward
- inspect forecastability diagnostics on the underlying asset
- compare forecasted vs realized returns for each horizon
- diagnose whether sign, magnitude, or confidence mapping is failing

Important: this is **not** shuffled k-fold cross-validation. Each forecast at time `t` is evaluated only against future returns that occur after `t`.

In [3]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'src').exists():
    for parent in [project_root, *project_root.parents]:
        if (parent / 'src').exists():
            project_root = parent
            break
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from securities_analysis.forecast_validation import (
    add_realized_horizon_returns,
    compute_forecast_diagnostics,
    horizon_columns,
    prepare_forecast_validation_frame,
)

plt.style.use('seaborn-v0_8-whitegrid')

In [4]:
# Optional override: set artifact_name to a specific folder under artifacts/backtests.
# Leave as None to auto-pick the latest SPY daily 2024 backtest artifact.
artifact_name = None

backtests_dir = project_root / 'artifacts' / 'backtests'
if artifact_name:
    artifact_dir = backtests_dir / artifact_name
else:
    candidates = sorted(backtests_dir.glob('SPY_day_2024-01-01_2024-12-31_*'))
    if not candidates:
        raise FileNotFoundError(f'No matching backtest artifacts found under {backtests_dir}')
    artifact_dir = candidates[-1]

steps_path = artifact_dir / 'steps.csv'
steps_frame = pd.read_csv(steps_path)
validation_frame = add_realized_horizon_returns(prepare_forecast_validation_frame(steps_frame))
horizon_map = horizon_columns(validation_frame)
metrics_frame, diagnostics_summary = compute_forecast_diagnostics(steps_frame)

artifact_dir, list(horizon_map.items())

(WindowsPath('c:/Users/Eddie/Programming/securities-analysis/artifacts/backtests/SPY_day_2024-01-01_2024-12-31_20260321_183154'),
 [(15, 'h15_expected_return'),
  (30, 'h30_expected_return'),
  (60, 'h60_expected_return')])

## Validation Method

This summary is written by the code so the artifact itself records the exact evaluation protocol.

In [5]:
pd.Series(diagnostics_summary['validation_method'])

name                                         rolling_origin_walk_forward
description            Each forecast at time t is produced using only...
uses_shuffled_folds                                                False
retrain_per_split                                                  False
notes                  This is appropriate for the current rule-based...
dtype: object

## Forecastability Diagnostics

These are lightweight diagnostics on the asset return stream itself. They do **not** prove forecastability, but they help us reason about whether the series looks closer to persistent structure or noisy near-random movement.

In [6]:
pd.Series(diagnostics_summary['forecastability']).sort_index()

lag1_autocorrelation    0.064576
spectral_entropy        0.920768
variance_ratio_20       0.454614
variance_ratio_5        1.065835
dtype: float64

## Horizon Metrics

These are the key forecast-quality metrics for each horizon. Correlation tells us whether the forecast magnitude moves in the same direction as realized future returns. Directional accuracy only checks the sign.

In [7]:
metrics_frame.sort_values('horizon_bars')

,horizon_bars,observations,forecast_column,correlation,mae,rmse,bias,directional_accuracy,directional_accuracy_nonzero,forecast_mean,realized_mean,forecast_std,realized_std
0,15,172,h15_expected_return,-0.443285,0.034587,0.046033,-0.001782,0.558140,0.558140,0.012456,0.014238,0.027908,0.026390
1,30,157,h30_expected_return,-0.594472,0.041952,0.051943,-0.006313,0.598726,0.598726,0.023832,0.030145,0.030591,0.027314
2,60,127,h60_expected_return,-0.434078,0.036373,0.046153,-0.009343,0.984252,0.984252,0.047411,0.056754,0.025376,0.028182


## Sign Diagnostics

The current model looked suspicious because directional accuracy was not terrible but correlation was strongly negative. This section checks whether the forecast sign or an inverted version of the sign matches future returns better.

In [8]:
rows = []
for horizon, forecast_column in horizon_map.items():
    realized_column = f'realized_return_h{horizon}'
    sample = validation_frame[[forecast_column, realized_column]].dropna()
    forecast_positive = sample[forecast_column] > 0
    realized_positive = sample[realized_column] > 0
    rows.append(
        {
            'horizon_bars': horizon,
            'sign_match_rate': (forecast_positive == realized_positive).mean(),
            'inverted_sign_match_rate': ((~forecast_positive) == realized_positive).mean(),
            'forecast_mean': sample[forecast_column].mean(),
            'realized_mean': sample[realized_column].mean(),
            'forecast_positive_rate': forecast_positive.mean(),
            'realized_positive_rate': realized_positive.mean(),
        }
    )

pd.DataFrame(rows)

,horizon_bars,sign_match_rate,inverted_sign_match_rate,forecast_mean,realized_mean,forecast_positive_rate,realized_positive_rate
0,15,0.558140,0.441860,0.012456,0.014238,0.750000,0.773256
1,30,0.598726,0.401274,0.023832,0.030145,0.751592,0.847134
2,60,0.984252,0.015748,0.047411,0.056754,0.992126,0.992126


## Forecast vs Realized Scatter

If the model is good, points should slope upward. If the slope looks downward, the forecast magnitude is systematically wrong.

In [ ]:
fig, axes = plt.subplots(len(horizon_map), 1, figsize=(8, 4 * len(horizon_map)))
if len(horizon_map) == 1:
    axes = [axes]

for ax, (horizon, forecast_column) in zip(axes, horizon_map.items()):
    realized_column = f'realized_return_h{horizon}'
    sample = validation_frame[[forecast_column, realized_column]].dropna()
    ax.scatter(sample[forecast_column], sample[realized_column], alpha=0.6)
    ax.axhline(0.0, color='black', linewidth=1, linestyle='--')
    ax.axvline(0.0, color='black', linewidth=1, linestyle='--')
    ax.set_title(f'Horizon {horizon}: Forecast vs Realized')
    ax.set_xlabel('Forecast return proxy')
    ax.set_ylabel('Realized future log return')

plt.tight_layout()
plt.show()

## Time-Series Overlay

This makes it easier to see whether the model is simply staying too bullish for too long.

In [ ]:
fig, axes = plt.subplots(len(horizon_map), 1, figsize=(12, 4 * len(horizon_map)), sharex=True)
if len(horizon_map) == 1:
    axes = [axes]

for ax, (horizon, forecast_column) in zip(axes, horizon_map.items()):
    realized_column = f'realized_return_h{horizon}'
    sample = validation_frame[['timestamp', forecast_column, realized_column]].dropna().copy()
    sample = sample.set_index('timestamp')
    sample[forecast_column].plot(ax=ax, label='forecast', linewidth=1.5)
    sample[realized_column].plot(ax=ax, label='realized', linewidth=1.2)
    ax.axhline(0.0, color='black', linewidth=1, linestyle='--')
    ax.set_title(f'Horizon {horizon}: Forecast vs Realized Through Time')
    ax.legend()

plt.tight_layout()
plt.show()

## Concrete Rows To Inspect

These rows make it obvious when short horizons disagree with long horizons but the aggregate stays positive.

In [ ]:
columns = [
    'timestamp',
    'close_price',
    'signal_confidence',
    'target_position',
    'h15_expected_return',
    'h30_expected_return',
    'h60_expected_return',
    'aggregate_score',
    'agreement_ratio',
    'regime_label',
    'realized_return_h15',
    'realized_return_h30',
    'realized_return_h60',
]
validation_frame[columns].head(15)

## Working Hypotheses

If the model remains weak, the likely failure modes are:
- the long horizon dominates too heavily and keeps the aggregate bullish
- expected return proxies are too naive because they use in-sample horizon means
- confidence is not aligned with realized opportunity
- horizon combination needs regime gating instead of always averaging

Next redesign ideas:
- use horizon z-scores or rank-normalized features instead of raw means
- require horizon agreement more strictly before taking size
- let the long horizon set regime and shorter horizons set timing
- evaluate forecast quality before changing the risk shell